# Build Exposure Variables And Analysis Dataset

Purpose: construct one thread-level analytic dataset containing combined provider, Physician/APP, other-human-HCW, system, and patient-initiated-thread exposures together with demographics, CCI, outcomes, and encounter covariates.

This notebook starts from the 30-day threaded message dataset created in `02_threading_diagnostics_sensitivity.ipynb`. It constructs all message-exposure families from the same thread anchors, adds message-time demographics/CCI variables from `04_build_demographics_cci_clean.ipynb`, adds pre-thread encounter covariates from the encounter timeline created in `03_build_visit_encounters_clean.ipynb`, and exports one unified analysis dataset.

## Scope

This notebook does:

- load `messages_with_threads.parquet`
- merge canonical demographics and message-time Charlson CCI handoff files
- define thread anchors and construct combined-provider and provider-type-specific exposure variables
- create patient-reply outcomes
- add pre-thread encounter covariates, including `log_encounters_past_7d`
- apply the manuscript analytic period and save `analysis_df_all_exposures_with_encounter_covariates.parquet`

This notebook does not:

- rebuild sender/source classification
- rebuild thread IDs
- rebuild the raw encounter timeline
- run regression models or generate figures
- display row-level patient or message examples

## Downstream Handoff

The final output `analysis_df_all_exposures_with_encounter_covariates.parquet` is used by `06_analysis_models_clean.ipynb` and `07_physapp_demographic_cci_interactions_clean.ipynb`.


## 1. Setup And Load Threaded Messages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "README.md").is_file():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = find_repo_root()
OUTPUT_ROOT = Path(os.environ.get("JMIR_REPRO_OUTPUT_DIR", REPO_ROOT / "outputs")).expanduser().resolve()
WORKING_DATA_DIR = Path(
    os.environ.get("JMIR_WORKING_DATA_DIR", OUTPUT_ROOT / "working_data")
).expanduser().resolve()
WORKING_DATA_DIR.mkdir(parents=True, exist_ok=True)


def require_input_file(file_name: str) -> Path:
    path = WORKING_DATA_DIR / file_name
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required pipeline handoff: {path}. Run the preceding notebooks first."
        )
    return path


MESSAGES_WITH_THREADS_PATH = require_input_file("messages_with_threads.parquet")
PATIENT_DEMOGRAPHICS_PATH = WORKING_DATA_DIR / "patient_demographics_clean.parquet"
MESSAGE_DEMOGRAPHICS_PATH = WORKING_DATA_DIR / "message_demographics_clean.parquet"
MESSAGE_CHARLSON_CCI_PATH = WORKING_DATA_DIR / "message_level_charlson_cci.parquet"
ENCOUNTERS_PATH = require_input_file("encounters_unified_clean.parquet")
ANALYSIS_OUTPUT_PATH = WORKING_DATA_DIR / "analysis_df_all_exposures_with_encounter_covariates.parquet"

all_messages_clean = pd.read_parquet(MESSAGES_WITH_THREADS_PATH, engine="pyarrow")

print("REPO_ROOT:", REPO_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("WORKING_DATA_DIR:", WORKING_DATA_DIR)
print("Loaded threaded messages:", MESSAGES_WITH_THREADS_PATH)
print("Shape:", all_messages_clean.shape)
print("Optional demographics handoff:", MESSAGE_DEMOGRAPHICS_PATH)
print("Optional Charlson handoff:", MESSAGE_CHARLSON_CCI_PATH)
print("Encounter handoff:", ENCOUNTERS_PATH)

## 2. Merge Demographics And Message-Time CCI

Demographic and Charlson CCI variables are built in the preceding demographics/CCI notebook and merged here before constructing the thread-level analytic dataset. Age is calculated at the message year, and Charlson CCI is calculated using diagnoses active at the message timestamp.

In [ ]:
covariate_handoffs = [
    PATIENT_DEMOGRAPHICS_PATH,
    MESSAGE_DEMOGRAPHICS_PATH,
    MESSAGE_CHARLSON_CCI_PATH,
]

if all(path.exists() for path in covariate_handoffs):
    patient_demographics = pd.read_parquet(PATIENT_DEMOGRAPHICS_PATH, engine="pyarrow")
    message_demographics = pd.read_parquet(MESSAGE_DEMOGRAPHICS_PATH, engine="pyarrow")
    message_charlson_cci = pd.read_parquet(MESSAGE_CHARLSON_CCI_PATH, engine="pyarrow")
    covariate_source = "separate demographics/CCI handoff files"
else:
    required_embedded_covariates = [
        "MESSAGE_ID", "DMRN", "BIRTH_YEAR", "Message_Year",
        "Age_at_Message", "AgeGroup65", "Gender", "Race",
        "Ethnicity", "Charlson_score", "Charlson_categories",
        "Charlson_group",
    ]
    missing_covariates = [
        col for col in required_embedded_covariates
        if col not in all_messages_clean.columns
    ]
    if missing_covariates:
        raise KeyError(
            "The threaded message file is missing required covariates and the "
            "separate demographics/CCI handoffs are unavailable: "
            + ", ".join(missing_covariates)
        )

    message_demographics = all_messages_clean[[
        "MESSAGE_ID", "BIRTH_YEAR", "Message_Year",
        "Age_at_Message", "AgeGroup65",
    ]].copy()
    message_charlson_cci = all_messages_clean[[
        "MESSAGE_ID", "Charlson_score", "Charlson_categories",
        "Charlson_group",
    ]].copy()
    patient_demographics = (
        all_messages_clean[["DMRN", "Gender", "Race", "Ethnicity"]]
        .drop_duplicates("DMRN")
        .rename(columns={"Race": "Race_Grouped", "Ethnicity": "Ethnicity_Grouped"})
    )
    covariate_source = "covariates embedded in the legacy threaded message file"

print("Covariate source:", covariate_source)

all_messages_clean["DMRN"] = all_messages_clean["DMRN"].astype("string").str.strip()
patient_demographics["DMRN"] = patient_demographics["DMRN"].astype("string").str.strip()

# Refresh covariate columns from the canonical demographics/CCI outputs.
covariate_cols_to_refresh = [
    "BIRTH_YEAR",
    "Message_Year",
    "Age_at_Message",
    "AgeGroup65",
    "Age65",
    "Age3cat",
    "Gender",
    "Race",
    "Race_Grouped",
    "Ethnicity",
    "Ethnicity_Grouped",
    "Charlson_score",
    "Charlson_categories",
    "Charlson_group",
]
all_messages_clean = all_messages_clean.drop(
    columns=[c for c in covariate_cols_to_refresh if c in all_messages_clean.columns],
    errors="ignore",
)

message_demo_cols = [
    c for c in ["MESSAGE_ID", "BIRTH_YEAR", "Message_Year", "Age_at_Message", "AgeGroup65"]
    if c in message_demographics.columns
]
message_demo_for_merge = message_demographics[message_demo_cols].drop_duplicates("MESSAGE_ID")

message_cci_cols = [
    c for c in ["MESSAGE_ID", "Charlson_score", "Charlson_categories", "Charlson_group"]
    if c in message_charlson_cci.columns
]
message_cci_for_merge = message_charlson_cci[message_cci_cols].drop_duplicates("MESSAGE_ID")

patient_demo_cols = [
    c for c in ["DMRN", "Gender", "Race_Grouped", "Ethnicity_Grouped"]
    if c in patient_demographics.columns
]
patient_demo_for_merge = (
    patient_demographics[patient_demo_cols]
    .drop_duplicates("DMRN")
    .rename(columns={"Race_Grouped": "Race", "Ethnicity_Grouped": "Ethnicity"})
)

all_messages_clean = all_messages_clean.merge(
    message_demo_for_merge,
    on="MESSAGE_ID",
    how="left",
    validate="many_to_one",
)
all_messages_clean = all_messages_clean.merge(
    message_cci_for_merge,
    on="MESSAGE_ID",
    how="left",
    validate="many_to_one",
)
all_messages_clean = all_messages_clean.merge(
    patient_demo_for_merge,
    on="DMRN",
    how="left",
    validate="many_to_one",
)

age_group65 = pd.to_numeric(all_messages_clean.get("AgeGroup65"), errors="coerce")
all_messages_clean["Age65"] = pd.Series(pd.NA, index=all_messages_clean.index, dtype="string")
all_messages_clean.loc[age_group65.eq(0), "Age65"] = "<65"
all_messages_clean.loc[age_group65.eq(1), "Age65"] = "\u226565"

age = pd.to_numeric(all_messages_clean["Age_at_Message"], errors="coerce")
all_messages_clean["Age3cat"] = pd.cut(
    age,
    bins=[-np.inf, 39, 64, np.inf],
    labels=["<40", "40-64", "\u226565"],
)

merge_qa = pd.DataFrame(
    [
        {"variable": col, "missing_n": int(all_messages_clean[col].isna().sum()), "missing_pct": all_messages_clean[col].isna().mean() * 100}
        for col in ["Age_at_Message", "Age65", "Age3cat", "Gender", "Race", "Ethnicity", "Charlson_group"]
        if col in all_messages_clean.columns
    ]
)

print("Rows after demographics/CCI merge:", len(all_messages_clean))
display(merge_qa)

## 3. Restrict To Analysis Columns

In [ ]:
# ======================================================
# 🧹 Keep only columns needed for analysis
# ======================================================

cols_to_keep = [
    'MESSAGE_ID',
    'DMRN',
    'CREATED_TIME',
    'MESG_TYPE',
    'Prov_DEPT_For_Appt',
    'SUBJECT_CLEAN',
    'is_system_generated',
    'thread_id',
    'Message_Year',
    'Age_at_Message',
    'Age65',
    'Gender',
    'Race',
    'Ethnicity',
    'Charlson_group',
    'Sender_all',
    'Receiver_all',
    'origin'
]

# Keep only those columns that exist in the DataFrame
all_messages_clean = all_messages_clean.loc[:, [c for c in cols_to_keep if c in all_messages_clean.columns]]

print("✅ Columns retained:", len(all_messages_clean.columns))
print(all_messages_clean.columns.tolist())

## 4. Patient And Thread Descriptive Summaries

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================
# 1️⃣ Collapse to patient level
# Each patient appears once, using their first message record
# ==============================================================
patient_df = (
    all_messages_clean.sort_values("CREATED_TIME")  # or equivalent timestamp
    .drop_duplicates(subset="DMRN", keep="first")  # 1 row per patient
)
print(f"Unique patients: {len(patient_df):,}")

# ==============================================================
# 2️⃣ Median [IQR] Age
# ==============================================================
median_age = patient_df['Age_at_Message'].median()
q1, q3 = patient_df['Age_at_Message'].quantile([0.25, 0.75])
print(f"Median age: {median_age:.1f} [IQR {q1:.1f}–{q3:.1f}]")

# ==============================================================
# 3️⃣ Gender distribution (n, %)
# ==============================================================
gender_counts = patient_df['Gender'].value_counts(dropna=False)
gender_pct = (gender_counts / len(patient_df) * 100).round(1)
gender_summary = pd.DataFrame({'n': gender_counts, '%': gender_pct})
print("\nGender Distribution:")
print(gender_summary)

# ==============================================================
# 4️⃣ Race distribution (n, %)
# ==============================================================
race_counts = patient_df['Race'].value_counts(dropna=False)
race_pct = (race_counts / len(patient_df) * 100).round(1)
race_summary = pd.DataFrame({'n': race_counts, '%': race_pct})
print("\nRace Distribution:")
print(race_summary)

# ==============================================================
# 5️⃣ Ethnicity distribution (n, %)
# ==============================================================
eth_counts = patient_df['Ethnicity'].value_counts(dropna=False)
eth_pct = (eth_counts / len(patient_df) * 100).round(1)
eth_summary = pd.DataFrame({'n': eth_counts, '%': eth_pct})
print("\nEthnicity Distribution:")
print(eth_summary)

# ==============================================================
# 6️⃣ Charlson Index groups (0, 1–2, 3–4, ≥5)
# ==============================================================
charlson_counts = patient_df['Charlson_group'].value_counts().sort_index()
charlson_pct = (charlson_counts / len(patient_df) * 100).round(1)
charlson_summary = pd.DataFrame({'n': charlson_counts, '%': charlson_pct})
print("\nCharlson Index (grouped):")
print(charlson_summary)

In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1️⃣ Messages per patient
# ============================================================
msg_per_patient = (
    all_messages_clean.groupby("DMRN")["MESSAGE_ID"].nunique()
)
median_msgs = msg_per_patient.median()
q1, q3 = msg_per_patient.quantile([0.25, 0.75])
print(f"Messages per patient: Median {median_msgs:.0f} [IQR {q1:.0f}–{q3:.0f}]")

# ============================================================
# 2️⃣ Threads per patient
# ============================================================
threads_per_patient = (
    all_messages_clean.groupby("DMRN")["thread_id"].nunique()
)
median_threads = threads_per_patient.median()
q1, q3 = threads_per_patient.quantile([0.25, 0.75])
print(f"Threads per patient: Median {median_threads:.0f} [IQR {q1:.0f}–{q3:.0f}]")

# ============================================================
# 3️⃣ Messages per thread
# ============================================================
msgs_per_thread = (
    all_messages_clean.groupby("thread_id")["MESSAGE_ID"].nunique()
)
median_msgs_thread = msgs_per_thread.median()
q1, q3 = msgs_per_thread.quantile([0.25, 0.75])
print(f"Messages per thread: Median {median_msgs_thread:.0f} [IQR {q1:.0f}–{q3:.0f}]")

# ============================================================
# 4️⃣ System-generated messages per patient
# ============================================================
system_msgs_per_patient = (
    all_messages_clean.query("Sender_all == 'System'")
    .groupby("DMRN")["MESSAGE_ID"].nunique()
)
median_sys = system_msgs_per_patient.median()
q1, q3 = system_msgs_per_patient.quantile([0.25, 0.75])
print(f"System-generated messages per patient: Median {median_sys:.0f} [IQR {q1:.0f}–{q3:.0f}]")

# ============================================================
# 5️⃣ Provider-generated messages per patient
# ============================================================
provider_categories = [
    "APP", "Administration", "Allied Health", "Behavioral Health",
    "Nurse", "Physician", "Student", "Support Staff",
    "Tech Specialist", "Therapist", "Unknown"
]

provider_msgs_per_patient = (
    all_messages_clean.query("Sender_all in @provider_categories")
    .groupby("DMRN")["MESSAGE_ID"].nunique()
)
median_prov = provider_msgs_per_patient.median()
q1, q3 = provider_msgs_per_patient.quantile([0.25, 0.75])
print(f"Provider-generated messages per patient: Median {median_prov:.0f} [IQR {q1:.0f}–{q3:.0f}]")

In [ ]:
# Show distinct categories in Sender_all with counts
sender_counts = all_messages_clean["Sender_all"].astype(str).value_counts(dropna=False)
print(f"Distinct Sender_all categories (n={sender_counts.size}):")
print(sender_counts)
# also expose the category list
sender_categories = sender_counts.index.tolist()
sender_categories

## 5. Build Thread Anchors And Prior Message Exposures

Anchors are HCW/provider-initiated threads. Prior exposure variables count provider messages, system messages, and patient-initiated threads within prespecified lookback windows.

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# FAST EXPOSURE BUILDER (vectorized per patient; no merge_asof)
# Anchors: human provider-initiated threads (exclude System initiators)
# Exposures in [t-window, t]  (INCLUDES events exactly at anchor time t)
#   1) provider msgs (all non-patient, non-system human staff)
#   2) Physician/APP msgs and other-human-HCW msgs as provider subgroups
#   3) system msgs (Sender_all == "System" OR is_system_generated == 1)
#   4) patient-initiated threads (thread starts) in prior window
#
# Outputs:
#   anchors: 1 row per anchor thread with exposure counts + log1p transforms
# ============================================================

# -----------------------------
# 0) Start from raw message-level df
# -----------------------------
df = all_messages_clean.copy()

df["CREATED_TIME"] = pd.to_datetime(df["CREATED_TIME"], errors="coerce")
df = df.dropna(subset=["DMRN", "thread_id", "CREATED_TIME"]).copy()

df["DMRN"] = df["DMRN"].astype(str)
df["thread_id"] = df["thread_id"].astype(str)

# If is_system_generated doesn't exist, create placeholder 0
if "is_system_generated" not in df.columns:
    df["is_system_generated"] = 0

physapp_set = {"Physician", "APP"}
is_system_sender = (
    df["Sender_all"].eq("System")
    | (df["is_system_generated"].fillna(0).astype(int) == 1)
)
df["sender_group"] = np.select(
    [
        df["Sender_all"].eq("Patient"),
        is_system_sender,
        df["Sender_all"].isin(physapp_set),
    ],
    ["Patient", "System", "PhysAPP"],
    default="OtherHuman",
)

# -----------------------------
# 1) Thread-first table (initiator sender + time + msg id)
# -----------------------------
# Tie-break: earliest CREATED_TIME, then smallest MESSAGE_ID (if present)
if "MESSAGE_ID" in df.columns:
    df["_msgid_sort"] = pd.to_numeric(df["MESSAGE_ID"], errors="coerce")
else:
    df["_msgid_sort"] = np.nan

df_sorted = df.sort_values(["thread_id", "CREATED_TIME", "_msgid_sort"], kind="mergesort")

keep_cols = ["thread_id", "DMRN", "CREATED_TIME", "Sender_all", "sender_group", "is_system_generated"]
if "MESSAGE_ID" in df.columns:
    keep_cols.append("MESSAGE_ID")

thread_first = (
    df_sorted.groupby("thread_id", as_index=False)
    .first()
    .loc[:, keep_cols]
    .rename(columns={
        "CREATED_TIME": "thread_first_time",
        "Sender_all": "thread_initiator_sender",
        "sender_group": "thread_initiator_group",
        "is_system_generated": "thread_initiator_is_system_generated",
        **({"MESSAGE_ID": "thread_first_message_id"} if "MESSAGE_ID" in df.columns else {})
    })
)

thread_first["thread_first_time"] = pd.to_datetime(thread_first["thread_first_time"], errors="coerce")
thread_first = thread_first.dropna(subset=["thread_first_time"]).copy()

# -----------------------------
# 2) Define anchors: human provider-initiated threads (exclude System initiators)
# -----------------------------
anchor_mask = thread_first["thread_initiator_group"].isin(["PhysAPP", "OtherHuman"])

anchor_cols = ["thread_id", "DMRN", "thread_first_time",
               "thread_initiator_sender", "thread_initiator_group"]
if "thread_first_message_id" in thread_first.columns:
    anchor_cols.append("thread_first_message_id")

anchors = thread_first.loc[anchor_mask, anchor_cols].copy()
anchors = anchors.sort_values(["DMRN", "thread_first_time"], kind="mergesort").reset_index(drop=True)

print(f"✅ Anchors (human provider-initiated; excluding system initiators): {len(anchors):,} threads")

# -----------------------------
# 3) Build event time arrays per patient
# -----------------------------
# Combined provider messages are the union of Physician/APP and other-human-HCW messages.
provider_msgs = df.loc[
    df["sender_group"].isin(["PhysAPP", "OtherHuman"]),
    ["DMRN", "CREATED_TIME"]
].copy()

physapp_msgs = df.loc[
    df["sender_group"].eq("PhysAPP"),
    ["DMRN", "CREATED_TIME"]
].copy()

other_human_msgs = df.loc[
    df["sender_group"].eq("OtherHuman"),
    ["DMRN", "CREATED_TIME"]
].copy()

system_msgs = df.loc[
    df["sender_group"].eq("System"),
    ["DMRN", "CREATED_TIME"]
].copy()

# patient-initiated thread starts (from thread_first)
patient_threads = thread_first.loc[
    thread_first["thread_initiator_group"].eq("Patient"),
    ["DMRN", "thread_first_time"]
].rename(columns={"thread_first_time": "CREATED_TIME"}).copy()

# Sort for searchsorted
provider_msgs = provider_msgs.sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")
physapp_msgs = physapp_msgs.sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")
other_human_msgs = other_human_msgs.sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")
system_msgs = system_msgs.sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")
patient_threads = patient_threads.sort_values(["DMRN", "CREATED_TIME"], kind="mergesort")

def build_time_dict(d):
    out = {}
    for pid, g in d.groupby("DMRN", sort=False):
        out[pid] = g["CREATED_TIME"].to_numpy(dtype="datetime64[ns]")  # already sorted
    return out

provider_times = build_time_dict(provider_msgs)
physapp_times = build_time_dict(physapp_msgs)
other_human_times = build_time_dict(other_human_msgs)
system_times = build_time_dict(system_msgs)
pat_thread_times = build_time_dict(patient_threads)

# -----------------------------
# 4) Vectorized window counts per patient
# -----------------------------
def count_in_window(times_sorted, t_arr, window_days):
    """
    counts events in [t - window, t]  (INCLUDES events exactly at t)
    """
    if times_sorted is None or len(times_sorted) == 0 or len(t_arr) == 0:
        return np.zeros(len(t_arr), dtype=np.int32)

    t_minus = t_arr - np.timedelta64(window_days, "D")
    left = np.searchsorted(times_sorted, t_minus, side="left")
    right = np.searchsorted(times_sorted, t_arr, side="right")  # include == t
    return (right - left).astype(np.int32)

# -----------------------------
# 4b) Compute exposure windows
# -----------------------------
windows = [7, 14, 30, 90]  # change if you want

for wd in windows:
    anchors[f"provider_msgs_past_{wd}d"] = 0
    anchors[f"physapp_msgs_past_{wd}d"] = 0
    anchors[f"other_human_msgs_past_{wd}d"] = 0
    anchors[f"system_msgs_past_{wd}d"] = 0
    anchors[f"patient_initiated_threads_past_{wd}d"] = 0

anchors["thread_first_time"] = pd.to_datetime(anchors["thread_first_time"], errors="coerce")
anchors = anchors.dropna(subset=["thread_first_time"]).copy()

for pid, g in anchors.groupby("DMRN", sort=False):
    idx = g.index.to_numpy()
    t = g["thread_first_time"].to_numpy(dtype="datetime64[ns]")

    pt_provider = provider_times.get(pid, None)
    pt_physapp = physapp_times.get(pid, None)
    pt_other_human = other_human_times.get(pid, None)
    pt_system = system_times.get(pid, None)
    pt_pthr = pat_thread_times.get(pid, None)

    for wd in windows:
        anchors.loc[idx, f"provider_msgs_past_{wd}d"] = count_in_window(pt_provider, t, wd)
        anchors.loc[idx, f"physapp_msgs_past_{wd}d"] = count_in_window(pt_physapp, t, wd)
        anchors.loc[idx, f"other_human_msgs_past_{wd}d"] = count_in_window(pt_other_human, t, wd)
        anchors.loc[idx, f"system_msgs_past_{wd}d"] = count_in_window(pt_system, t, wd)
        anchors.loc[idx, f"patient_initiated_threads_past_{wd}d"] = count_in_window(pt_pthr, t, wd)

# -----------------------------
# 5) Transforms for models (log1p + squares)
# -----------------------------
for wd in windows:
    anchors[f"log_provider_msgs_past_{wd}d"] = np.log1p(anchors[f"provider_msgs_past_{wd}d"].astype(float))
    anchors[f"log_provider_msgs_sq_{wd}d"]   = anchors[f"log_provider_msgs_past_{wd}d"] ** 2

    anchors[f"log_physapp_msgs_past_{wd}d"] = np.log1p(anchors[f"physapp_msgs_past_{wd}d"].astype(float))
    anchors[f"log_physapp_msgs_sq_{wd}d"] = anchors[f"log_physapp_msgs_past_{wd}d"] ** 2

    anchors[f"log_other_human_msgs_past_{wd}d"] = np.log1p(anchors[f"other_human_msgs_past_{wd}d"].astype(float))
    anchors[f"log_other_human_msgs_sq_{wd}d"] = anchors[f"log_other_human_msgs_past_{wd}d"] ** 2

    anchors[f"log_system_msgs_past_{wd}d"] = np.log1p(anchors[f"system_msgs_past_{wd}d"].astype(float))
    anchors[f"log_system_msgs_sq_{wd}d"]   = anchors[f"log_system_msgs_past_{wd}d"] ** 2

    anchors[f"log_patient_initiated_threads_past_{wd}d"] = np.log1p(
        anchors[f"patient_initiated_threads_past_{wd}d"].astype(float)
    )
    anchors[f"log_patient_initiated_threads_sq_{wd}d"] = anchors[f"log_patient_initiated_threads_past_{wd}d"] ** 2

# -----------------------------
# 6) Exposure QA
# -----------------------------
for wd in windows:
    combined = anchors[f"provider_msgs_past_{wd}d"]
    split_sum = (
        anchors[f"physapp_msgs_past_{wd}d"]
        + anchors[f"other_human_msgs_past_{wd}d"]
    )
    assert combined.equals(split_sum), (
        f"Combined provider counts do not equal provider subgroup counts for {wd} days"
    )

print("Exposure QA passed: combined provider counts equal Physician/APP plus other-human-HCW counts.")
display(
    anchors[[
        "provider_msgs_past_30d",
        "physapp_msgs_past_30d",
        "other_human_msgs_past_30d",
        "system_msgs_past_30d",
        "patient_initiated_threads_past_30d",
    ]].describe()
)

# anchors is now your thread-level analysis frame with exposures.

## 6. Create Patient Reply Outcomes

In [ ]:
## Create outcome varaibles

import pandas as pd
import numpy as np

# msgs: message-level
msgs = all_messages_clean.copy()
msgs["CREATED_TIME"] = pd.to_datetime(msgs["CREATED_TIME"], errors="coerce")

# threads: your anchor thread-level df
thr = anchors.copy()
thr["thread_first_time"] = pd.to_datetime(thr["thread_first_time"], errors="coerce")

# 1) keep patient messages only, and merge on thread start time
patient = msgs.loc[msgs["Sender_all"].eq("Patient"), ["thread_id", "CREATED_TIME"]].copy()

#    (String is safest)
thr["thread_id"]  = thr["thread_id"].astype(str)
patient["thread_id"] = patient["thread_id"].astype(str)

patient = patient.merge(
    thr[["thread_id", "thread_first_time"]],
    on="thread_id",
    how="inner"
)

# 2) allow exact time: patient message at or after thread start
patient = patient.loc[patient["CREATED_TIME"] >= patient["thread_first_time"]]

# 3) first patient reply time per thread
first_reply = (
    patient.groupby("thread_id", as_index=False)["CREATED_TIME"]
    .min()
    .rename(columns={"CREATED_TIME": "first_patient_reply_time"})
)

# 4) merge back to thread-level and build outcomes
thr = thr.merge(first_reply, on="thread_id", how="left")

thr["patient_replied_7d"] = (
    thr["first_patient_reply_time"].notna() &
    (thr["first_patient_reply_time"] <= thr["thread_first_time"] + pd.Timedelta(days=7))
).astype(int)

thr["patient_replied_14d"] = (
    thr["first_patient_reply_time"].notna() &
    (thr["first_patient_reply_time"] <= thr["thread_first_time"] + pd.Timedelta(days=14))
).astype(int)

thr["patient_replied_30d"] = (
    thr["first_patient_reply_time"].notna() &
    (thr["first_patient_reply_time"] <= thr["thread_first_time"] + pd.Timedelta(days=30))
).astype(int)

thr["patient_replied_ever"] = thr["first_patient_reply_time"].notna().astype(int)

# optional: delay (days)
thr["reply_delay_days"] = (
    (thr["first_patient_reply_time"] - thr["thread_first_time"])
    .dt.total_seconds() / (3600*24)
)

# 7) Quick sanity checks
print("Threads in anchors:", len(thr))
print("Reply 7d:", thr["patient_replied_7d"].mean())
print("Reply 14d:", thr["patient_replied_14d"].mean())
print("Reply 30d:", thr["patient_replied_30d"].mean())
print("Reply ever:", thr["patient_replied_ever"].mean())

# thr is now your analysis df with exposures + outcomes
# If you want to overwrite anchors:
anchors_with_outcomes = thr

## 7. Add Patient-Level Demographic And Comorbidity Covariates

In [ ]:
import pandas as pd
import numpy as np

# --- helper: mode (most frequent) ---
def mode_or_first(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]

# 1) Build patient-level covariate table (1 row per DMRN)
cov_cols_cat = [c for c in ["Age_at_Message", "Age65", "Age3cat", "Gender", "Race", "Ethnicity", "Charlson_group"] if c in all_messages_clean.columns]

patient_cov = (
    all_messages_clean
    .dropna(subset=["DMRN"])
    .assign(DMRN=lambda d: d["DMRN"].astype(str))
    .groupby("DMRN", as_index=False)
    .agg(
        **{c: (c, mode_or_first) for c in cov_cols_cat},
    )
)

# 2) Merge onto anchors_with_outcomes
anchors_with_outcomes = anchors_with_outcomes.copy()
anchors_with_outcomes["DMRN"] = anchors_with_outcomes["DMRN"].astype(str)

analysis_df = anchors_with_outcomes.merge(patient_cov, on="DMRN", how="left")

# 3) quick checks
print("Rows (anchors):", len(anchors_with_outcomes))
print("Rows (analysis_df):", len(analysis_df))
print("Missingness snapshot:")
missingness_cols = [c for c in ["Age65", "Age3cat", "Gender", "Race", "Ethnicity", "Charlson_group"] if c in analysis_df.columns]
print(analysis_df[missingness_cols].isna().mean().round(4))

# E. save the “all-in-one” file

## 8. Load And Prepare Encounter Timeline

In [ ]:
# read the cleaned encounters data
encounters_unified_clean = pd.read_parquet(ENCOUNTERS_PATH)

In [ ]:
cols_to_drop = [
    "SOURCE",
    "SOURCE_DETAIL",
    "visit_type_detailed",
    "encounter_context_primary",
    "encounter_context_primary_source",
    "encounter_department_specialty",
    "encounter_reason_or_dx_text_combined",
]

encounters_clean = encounters_unified_clean.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
# ============================================================
# Encounter covariates: prepare encounter data
# ============================================================

import numpy as np
import pandas as pd

# Assumes:
#   analysis_df has: DMRN, thread_first_time
#   encounters_clean has: DMRN or patient_id, encounter_start, encounter_type or visit_type_grouped

enc = encounters_clean.copy()

# Standardize patient ID column
if "patient_id" in enc.columns and "DMRN" not in enc.columns:
    enc = enc.rename(columns={"patient_id": "DMRN"})

# Standardize encounter datetime column
if "ENCOUNTER_START" in enc.columns and "encounter_start" not in enc.columns:
    enc = enc.rename(columns={"ENCOUNTER_START": "encounter_start"})

# Standardize encounter type column
if "encounter_type" not in enc.columns:
    if "TYPE_GROUP" in enc.columns:
        enc = enc.rename(columns={"TYPE_GROUP": "encounter_type"})
    elif "visit_type_grouped" in enc.columns:
        enc = enc.rename(columns={"visit_type_grouped": "encounter_type"})

enc["encounter_start"] = pd.to_datetime(enc["encounter_start"], errors="coerce")
analysis_df["thread_first_time"] = pd.to_datetime(analysis_df["thread_first_time"], errors="coerce")

# Keep valid encounter rows only
enc = enc.dropna(subset=["DMRN", "encounter_start"]).copy()

# Keep encounters only before or during the study period represented in analysis_df
enc = enc[enc["encounter_start"] <= analysis_df["thread_first_time"].max()].copy()

# Normalize encounter type
enc["encounter_type_norm"] = (
    enc["encounter_type"]
    .astype("string")
    .str.lower()
    .str.strip()
)

print("Encounter rows:", enc.shape[0])
print("Unique encounter patients:", enc["DMRN"].nunique())
print("Analysis rows:", analysis_df.shape[0])
print("Unique analysis patients:", analysis_df["DMRN"].nunique())

display(enc["encounter_type_norm"].value_counts(dropna=False))

In [ ]:
# Define encounter type groups
# ============================================================

def classify_encounter_type(x):
    if pd.isna(x):
        return "unknown"
    
    x = str(x).lower()
    
    if "ed" in x or "emergency" in x:
        return "ed"
    elif "inpatient" in x or "hospital" in x:
        return "hospital"
    elif "clinic" in x or "office" in x or "ambulatory" in x or "outpatient" in x:
        return "clinic"
    else:
        return "other"

enc["encounter_group"] = enc["encounter_type_norm"].apply(classify_encounter_type)

display(enc["encounter_group"].value_counts(dropna=False))

## 9. Add Pre-Thread Encounter Covariates

In [ ]:
# ============================================================
# Faster pre-thread encounter covariates
# ============================================================

import numpy as np
import pandas as pd

def add_encounter_counts_fast(
    threads_df,
    encounters_df,
    windows=(7, 30),
    patient_col="DMRN",
    thread_time_col="thread_first_time",
    encounter_time_col="encounter_start",
):
    df = threads_df.copy()
    enc = encounters_df.copy()

    df[thread_time_col] = pd.to_datetime(df[thread_time_col], errors="coerce")
    enc[encounter_time_col] = pd.to_datetime(enc[encounter_time_col], errors="coerce")

    df["_row_id"] = np.arange(len(df))
    df = df.sort_values([patient_col, thread_time_col]).reset_index(drop=True)

    enc = enc.dropna(subset=[patient_col, encounter_time_col]).copy()
    enc = enc.sort_values([patient_col, encounter_time_col]).reset_index(drop=True)

    # Ensure encounter_group exists
    if "encounter_group" not in enc.columns:
        raise ValueError("encounters_df must contain encounter_group")

    for w in windows:
        for prefix in ["encounters", "clinic_encounters", "ed_encounters", "hospital_encounters", "other_encounters"]:
            df[f"{prefix}_past_{w}d"] = 0
        df[f"had_encounter_past_{w}d"] = 0

    # Pre-group encounters once
    enc_groups = {pid: g for pid, g in enc.groupby(patient_col, sort=False)}

    updated_parts = []

    for pid, g_threads in df.groupby(patient_col, sort=False):
        g_threads = g_threads.copy()
        g_enc = enc_groups.get(pid)

        if g_enc is None or g_enc.empty:
            updated_parts.append(g_threads)
            continue

        et = g_enc[encounter_time_col].to_numpy(dtype="datetime64[ns]")
        tt = g_threads[thread_time_col].to_numpy(dtype="datetime64[ns]")

        enc_type = g_enc["encounter_group"].astype("string").to_numpy()

        is_clinic = (enc_type == "clinic").astype(int)
        is_ed = (enc_type == "ed").astype(int)
        is_hospital = (enc_type == "hospital").astype(int)
        is_other = (enc_type == "other").astype(int)

        cum_total = np.r_[0, np.ones(len(et), dtype=int).cumsum()]
        cum_clinic = np.r_[0, is_clinic.cumsum()]
        cum_ed = np.r_[0, is_ed.cumsum()]
        cum_hospital = np.r_[0, is_hospital.cumsum()]
        cum_other = np.r_[0, is_other.cumsum()]

        right = np.searchsorted(et, tt, side="left")

        for w in windows:
            left = np.searchsorted(et, tt - np.timedelta64(w, "D"), side="left")

            total = cum_total[right] - cum_total[left]
            clinic = cum_clinic[right] - cum_clinic[left]
            ed = cum_ed[right] - cum_ed[left]
            hospital = cum_hospital[right] - cum_hospital[left]
            other = cum_other[right] - cum_other[left]

            g_threads[f"encounters_past_{w}d"] = total
            g_threads[f"clinic_encounters_past_{w}d"] = clinic
            g_threads[f"ed_encounters_past_{w}d"] = ed
            g_threads[f"hospital_encounters_past_{w}d"] = hospital
            g_threads[f"other_encounters_past_{w}d"] = other
            g_threads[f"had_encounter_past_{w}d"] = (total > 0).astype(int)
            g_threads[f"log_encounters_past_{w}d"] = np.log1p(total)

        updated_parts.append(g_threads)

    out = (
        pd.concat(updated_parts, axis=0)
        .sort_values("_row_id")
        .drop(columns="_row_id")
        .reset_index(drop=True)
    )

    # Derive log counts after all patient groups are combined so patients
    # without encounter records correctly retain log1p(0) = 0.
    for w in windows:
        out[f"log_encounters_past_{w}d"] = np.log1p(
            out[f"encounters_past_{w}d"].astype(float)
        )

    return out


analysis_df = add_encounter_counts_fast(
    threads_df=analysis_df,
    encounters_df=enc,
    windows=(7, 30),
)

for w in (7, 30):
    count_col = f"encounters_past_{w}d"
    log_col = f"log_encounters_past_{w}d"
    assert analysis_df[count_col].notna().all(), f"Missing values in {count_col}"
    assert analysis_df[log_col].notna().all(), f"Missing values in {log_col}"
    assert np.allclose(
        analysis_df[log_col],
        np.log1p(analysis_df[count_col].astype(float)),
    ), f"{log_col} is inconsistent with {count_col}"

print("Encounter covariate QA passed: count and log-count variables are complete and consistent.")

new_cols = [c for c in analysis_df.columns if "encounter" in c]
display(analysis_df[new_cols].describe().T)

In [ ]:
# Encounter covariate diagnostics
# ============================================================

encounter_covars = [
    "encounters_past_7d",
    "log_encounters_past_7d",
    "had_encounter_past_7d",
    "clinic_encounters_past_7d",
    "ed_encounters_past_7d",
    "hospital_encounters_past_7d",
    "other_encounters_past_7d",
    "encounters_past_30d",
    "log_encounters_past_30d",
    "had_encounter_past_30d",
    "clinic_encounters_past_30d",
    "ed_encounters_past_30d",
    "hospital_encounters_past_30d",
    "other_encounters_past_30d",
]

summary = []

for col in encounter_covars:
    if col in analysis_df.columns:
        s = analysis_df[col]
        summary.append({
            "variable": col,
            "mean": s.mean(),
            "median": s.median(),
            "p75": s.quantile(0.75),
            "p90": s.quantile(0.90),
            "p95": s.quantile(0.95),
            "max": s.max(),
            "zero_pct": (s == 0).mean() * 100,
        })

encounter_summary = pd.DataFrame(summary)

display(encounter_summary)

## 10. Export Final Analysis Dataset

In [ ]:
# ============================================================
# Apply the manuscript cohort restriction and save the canonical analysis dataset
# ============================================================

ANALYTIC_START = pd.Timestamp("2021-03-31")
ANALYTIC_END = pd.Timestamp("2023-11-30 23:59:59.999999999")

rows_before_restriction = len(analysis_df)
patients_before_restriction = analysis_df["DMRN"].nunique()
rows_missing_age_before_restriction = int(analysis_df["Age_at_Message"].isna().sum())
patients_missing_age_before_restriction = int(
    analysis_df.loc[analysis_df["Age_at_Message"].isna(), "DMRN"].nunique()
)
analysis_df["thread_first_time"] = pd.to_datetime(
    analysis_df["thread_first_time"], errors="coerce"
)
analysis_df = analysis_df.dropna(
    subset=["Age_at_Message", "Age65", "thread_first_time"]
).copy()
analysis_df = analysis_df.loc[
    analysis_df["thread_first_time"].between(
        ANALYTIC_START, ANALYTIC_END, inclusive="both"
    )
].copy()

assert analysis_df["Age_at_Message"].notna().all()
assert analysis_df["Age65"].notna().all()
assert analysis_df["thread_first_time"].between(
    ANALYTIC_START, ANALYTIC_END, inclusive="both"
).all()

analytic_sample_qa = pd.DataFrame([{
    "rows_before_restriction": int(rows_before_restriction),
    "rows_saved": int(len(analysis_df)),
    "rows_removed": int(rows_before_restriction - len(analysis_df)),
    "patients_before_restriction": int(patients_before_restriction),
    "rows_missing_age_before_restriction": rows_missing_age_before_restriction,
    "patients_missing_age_before_restriction": patients_missing_age_before_restriction,
    "patients_saved": int(analysis_df["DMRN"].nunique()),
    "analytic_start": analysis_df["thread_first_time"].min(),
    "analytic_end": analysis_df["thread_first_time"].max(),
}])
display(analytic_sample_qa)

analysis_df.to_parquet(ANALYSIS_OUTPUT_PATH, index=False)
print("Saved final analysis dataset:", ANALYSIS_OUTPUT_PATH)

## Reviewer CA #5: Other HCW Role Composition Table

This supplementary table is generated after the final thread-level analytic dataset has been created.


In [ ]:
# CHUNK 1: OTHER HCW ROLE COMPOSITION TABLE
# Target notebook: 05_build_exposure_analysis_dataset_clean.ipynb
# ======================================================================

from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side


ROLE_TABLE_TITLE = (
    "Table S9. Composition and 7-Day Patient Reply Rates of "
    "Other HCW-Initiated Threads by Role"
)

ROLE_TABLE_NOTE = (
    "Role categories were derived from the structured provider-role mapping "
    "used to define the physician/APP boundary. The physician category "
    "included physicians, residents, fellows, anesthesiologists, and doctors "
    "of osteopathic medicine. The APP category included nurse practitioners, "
    "nurse midwives, certified registered nurse anesthetists, physician "
    "assistants, advanced practice providers, and clinical nurse specialists. "
    "Other HCW comprised all remaining nonpatient, nonsystem health care "
    "worker roles. Percentages use all Other HCW-initiated threads as the "
    "denominator. Patient reply was defined as the first patient reply within "
    "7 days after thread initiation. HCW indicates health care worker; "
    "APP, advanced practice provider."
)

required_thread_columns = {
    "thread_id",
    "thread_first_message_id",
    "thread_initiator_group",
    "patient_replied_7d",
}

required_message_columns = {
    "MESSAGE_ID",
    "SENDER_CATEGORY",
}

missing_thread_columns = (
    required_thread_columns - set(analysis_df.columns)
)
missing_message_columns = (
    required_message_columns - set(all_messages_clean.columns)
)

if missing_thread_columns:
    raise KeyError(
        "analysis_df is missing required columns: "
        + ", ".join(sorted(missing_thread_columns))
    )

if missing_message_columns:
    raise KeyError(
        "all_messages_clean is missing required columns: "
        + ", ".join(sorted(missing_message_columns))
    )

thread_sample = analysis_df.copy()

if thread_sample["thread_id"].duplicated().any():
    raise AssertionError(
        "analysis_df must contain one row per HCW-initiated thread."
    )

thread_sample["thread_first_message_id"] = (
    thread_sample["thread_first_message_id"]
    .astype("string")
    .str.strip()
)

message_role_columns = ["MESSAGE_ID", "SENDER_CATEGORY"]
if "SENDER_TYPE" in all_messages_clean.columns:
    message_role_columns.append("SENDER_TYPE")
elif "Sender_all" in all_messages_clean.columns:
    message_role_columns.append("Sender_all")

message_role_lookup = (
    all_messages_clean[message_role_columns]
    .copy()
)
if "SENDER_TYPE" not in message_role_lookup.columns:
    message_role_lookup["SENDER_TYPE"] = message_role_lookup.get(
        "Sender_all",
        message_role_lookup["SENDER_CATEGORY"],
    )

message_role_lookup["MESSAGE_ID"] = (
    message_role_lookup["MESSAGE_ID"]
    .astype("string")
    .str.strip()
)

message_role_lookup = (
    message_role_lookup
    .drop_duplicates("MESSAGE_ID", keep="first")
    .rename(columns={
        "MESSAGE_ID": "thread_first_message_id",
        "SENDER_CATEGORY": "initiator_role_bucket",
        "SENDER_TYPE": "initiator_detailed_role",
    })
)

other_hcw_threads = (
    thread_sample.loc[
        thread_sample["thread_initiator_group"].eq("OtherHuman"),
        [
            "thread_id",
            "thread_first_message_id",
            "patient_replied_7d",
        ],
    ]
    .merge(
        message_role_lookup,
        on="thread_first_message_id",
        how="left",
        validate="many_to_one",
    )
)

other_hcw_threads["initiator_role_bucket"] = (
    other_hcw_threads["initiator_role_bucket"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Unknown")
)

other_hcw_threads["patient_replied_7d"] = (
    pd.to_numeric(
        other_hcw_threads["patient_replied_7d"],
        errors="raise",
    )
    .astype(int)
)

unexpected_roles = set(
    other_hcw_threads["initiator_role_bucket"].dropna()
) & {
    "Physician",
    "APP",
    "Patient",
    "System",
}

if unexpected_roles:
    raise AssertionError(
        "Roles outside the Other HCW definition were found: "
        + ", ".join(sorted(unexpected_roles))
    )

n_other_hcw_threads = len(other_hcw_threads)

if n_other_hcw_threads == 0:
    raise ValueError("No Other HCW-initiated threads were found.")

role_summary = (
    other_hcw_threads
    .groupby(
        "initiator_role_bucket",
        dropna=False,
        observed=True,
    )
    .agg(
        number_of_threads=("thread_id", "nunique"),
        patient_replied_7d_n=("patient_replied_7d", "sum"),
    )
    .reset_index()
    .rename(columns={
        "initiator_role_bucket": "Role",
    })
)

role_summary["percentage_of_other_hcw_threads"] = (
    100
    * role_summary["number_of_threads"]
    / n_other_hcw_threads
)

role_summary["patient_replied_7d_pct"] = (
    100
    * role_summary["patient_replied_7d_n"]
    / role_summary["number_of_threads"]
)

role_summary = (
    role_summary
    .sort_values(
        ["number_of_threads", "Role"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

if role_summary["number_of_threads"].sum() != n_other_hcw_threads:
    raise AssertionError(
        "Role-specific thread counts do not sum to all Other HCW threads."
    )

if not abs(
    role_summary["percentage_of_other_hcw_threads"].sum() - 100
) < 1e-8:
    raise AssertionError(
        "Role percentages do not sum to 100%."
    )

other_hcw_role_table = pd.DataFrame({
    "Role": role_summary["Role"],
    "Number of HCW-initiated threads":
        role_summary["number_of_threads"].map(
            lambda value: f"{int(value):,}"
        ),
    "Percentage of Other HCW threads":
        role_summary["percentage_of_other_hcw_threads"].map(
            lambda value: f"{value:.1f}%"
        ),
    "Patient reply within 7 days, n (%)": [
        (
            f"{int(n):,} ({pct:.1f}%)"
        )
        for n, pct in zip(
            role_summary["patient_replied_7d_n"],
            role_summary["patient_replied_7d_pct"],
        )
    ],
})

total_reply_n = int(
    other_hcw_threads["patient_replied_7d"].sum()
)
total_reply_pct = (
    100 * total_reply_n / n_other_hcw_threads
)

total_row = pd.DataFrame([{
    "Role": "All Other HCW roles",
    "Number of HCW-initiated threads":
        f"{n_other_hcw_threads:,}",
    "Percentage of Other HCW threads": "100.0%",
    "Patient reply within 7 days, n (%)":
        f"{total_reply_n:,} ({total_reply_pct:.1f}%)",
}])

other_hcw_role_table = pd.concat(
    [other_hcw_role_table, total_row],
    ignore_index=True,
)

role_table_styled = (
    other_hcw_role_table.style
    .hide(axis="index")
    .set_properties(
        subset=["Role"],
        **{"text-align": "left"},
    )
    .set_properties(
        subset=[
            "Number of HCW-initiated threads",
            "Percentage of Other HCW threads",
            "Patient reply within 7 days, n (%)",
        ],
        **{"text-align": "right"},
    )
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("border-top", "1.5px solid black"),
                ("border-bottom", "1px solid black"),
                ("font-weight", "bold"),
                ("padding", "6px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("border", "none"),
                ("padding", "6px"),
            ],
        },
        {
            "selector": "tbody tr:last-child td",
            "props": [
                ("border-top", "1px solid black"),
                ("border-bottom", "1.5px solid black"),
                ("font-weight", "bold"),
            ],
        },
    ])
)

display(HTML(f"<p><strong>{ROLE_TABLE_TITLE}</strong></p>"))
display(role_table_styled)
display(HTML(f"<p><em>Note.</em> {ROLE_TABLE_NOTE}</p>"))

ROLE_TABLE_OUTPUT_DIR = (
    RUN_DIR / "tables"
    if "RUN_DIR" in globals()
    else Path.cwd()
)
ROLE_TABLE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ROLE_TABLE_XLSX = (
    ROLE_TABLE_OUTPUT_DIR
    / "table_S9_other_hcw_role_composition.xlsx"
)
ROLE_TABLE_HTML = (
    ROLE_TABLE_OUTPUT_DIR
    / "table_S9_other_hcw_role_composition.html"
)

with pd.ExcelWriter(
    ROLE_TABLE_XLSX,
    engine="openpyxl",
) as writer:
    other_hcw_role_table.to_excel(
        writer,
        sheet_name="Table S9",
        startrow=2,
        index=False,
    )

    worksheet = writer.book["Table S9"]

    worksheet.merge_cells("A1:D1")
    worksheet["A1"] = ROLE_TABLE_TITLE
    worksheet["A1"].font = Font(
        name="Arial",
        size=10,
        bold=True,
    )
    worksheet["A1"].alignment = Alignment(
        wrap_text=True,
        vertical="top",
    )

    header_row = 3

    for cell in worksheet[header_row]:
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
        )
        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="D9E1F2",
        )
        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True,
        )

    final_data_row = (
        header_row + len(other_hcw_role_table)
    )

    thin_border = Side(
        style="thin",
        color="000000",
    )

    for cell in worksheet[header_row]:
        cell.border = Border(
            top=thin_border,
            bottom=thin_border,
        )

    for cell in worksheet[final_data_row]:
        cell.font = Font(
            name="Arial",
            size=10,
            bold=True,
        )
        cell.border = Border(
            top=thin_border,
            bottom=thin_border,
        )

    note_row = final_data_row + 2
    worksheet.merge_cells(
        start_row=note_row,
        start_column=1,
        end_row=note_row,
        end_column=4,
    )
    worksheet.cell(
        row=note_row,
        column=1,
        value=f"Note. {ROLE_TABLE_NOTE}",
    )
    worksheet.cell(
        row=note_row,
        column=1,
    ).font = Font(
        name="Arial",
        size=10,
    )
    worksheet.cell(
        row=note_row,
        column=1,
    ).alignment = Alignment(
        wrap_text=True,
        vertical="top",
    )

    worksheet.column_dimensions["A"].width = 30
    worksheet.column_dimensions["B"].width = 24
    worksheet.column_dimensions["C"].width = 25
    worksheet.column_dimensions["D"].width = 32
    worksheet.row_dimensions[1].height = 30
    worksheet.row_dimensions[note_row].height = 90
    worksheet.freeze_panes = "A4"
    worksheet.sheet_view.showGridLines = False

role_table_html = (
    f"<p><strong>{ROLE_TABLE_TITLE}</strong></p>"
    + role_table_styled.to_html()
    + f"<p><em>Note.</em> {ROLE_TABLE_NOTE}</p>"
)

ROLE_TABLE_HTML.write_text(
    role_table_html,
    encoding="utf-8",
)

print("Saved:", ROLE_TABLE_XLSX)
print("Saved:", ROLE_TABLE_HTML)


# ======================================================================
